# MP-0：リアルタイム複数顔・表情認識

- 学籍番号：M25W0243
- 顔検出：OpenCV YuNet（5点顔ランドマーク）
- 表情分類：EmotiEffLib/HSEmotion EfficientNet-B0 ONNX
- 顔動作：MediaPipe Face Landmarker（478点・52 blendshapes）
- 対応：顔の位置合わせ、複数人、EMA平滑化、判定不能ゲート、日本語表示、画像保存

カメラ映像から顔を検出し、5点ランドマークで224×224に位置合わせする。信頼できる場合のみ8種類の表情を表示し、MediaPipeから口の開き、眉上げ、眉の収縮、笑顔、目の見開きも数値化する。低品質や不確定な顔は「判定不能」とする。

## 1. ライブラリとモデルの確認

このNotebookは `MP-0` フォルダを作業フォルダとして開いてから実行する。

In [4]:
from pathlib import Path
from importlib.metadata import version
import sys

import cv2
import numpy as np
from PIL import Image, ImageDraw, ImageFont

PROJECT_DIR = Path.cwd()
if not (PROJECT_DIR / 'emotion_recognition.py').exists():
    raise FileNotFoundError('MP-0フォルダを作業フォルダとしてNotebookを開いてください。')

if str(PROJECT_DIR) not in sys.path:
    sys.path.insert(0, str(PROJECT_DIR))

MODEL_FILES = [
    PROJECT_DIR / 'models/face_detection_yunet_2023mar.onnx',
    PROJECT_DIR / 'models/enet_b0_8_best_vgaf.onnx',
    PROJECT_DIR / 'models/face_landmarker.task',
]

for model_path in MODEL_FILES:
    print(('OK  ' if model_path.exists() else 'NG  '), model_path.name)

print('Python:', sys.executable)
print('OpenCV:', cv2.__version__, cv2.__file__)
print('MediaPipe:', version('mediapipe'))
print('NumPy:', np.__version__)

OK   face_detection_yunet_2023mar.onnx
OK   enet_b0_8_best_vgaf.onnx
OK   face_landmarker.task
Python: /usr/local/bin/python3
OpenCV: 4.13.0 /Library/Frameworks/Python.framework/Versions/3.13/lib/python3.13/site-packages/cv2/__init__.py
MediaPipe: 0.10.35
NumPy: 2.2.2


## 2. OpenCV DNNモデルの読み込み

実装本体は `emotion_recognition.py` に分離している。次のセルで最新の実装を再読み込み、顔検出モデルと表情分類モデル、利用できる表情クラスを確認する。

In [5]:
import importlib
import emotion_recognition
importlib.reload(emotion_recognition)

from emotion_recognition import (
    EMOTION_LABELS_EN,
    EMOTION_LABELS_JA,
    EmotionRecognitionApp,
    run_camera_in_notebook,
)

# YuNet、EfficientNet-B0、MediaPipe Face Landmarkerを読み込む
app_test = EmotionRecognitionApp()
print('顔検出・表情分類・478点Face Landmarkerを読み込みました。')
for index, (english, japanese) in enumerate(zip(EMOTION_LABELS_EN, EMOTION_LABELS_JA)):
    print(f'{index}: {english} / {japanese}')
app_test.close()

W0000 00:00:1784278529.168902 4259736 face_landmarker_graph.cc:180] Sets FaceBlendshapesGraph acceleration to xnnpack by default.
I0000 00:00:1784278529.175847 4259736 gl_context.cc:407] GL version: 2.1 (2.1 Metal - 90.5), renderer: Apple M1 Pro
W0000 00:00:1784278529.177305 4259739 inference_feedback_manager.cc:121] Feedback manager requires a model with a single signature inference. Disabling support for feedback tensors.
W0000 00:00:1784278529.189469 4259743 inference_feedback_manager.cc:121] Feedback manager requires a model with a single signature inference. Disabling support for feedback tensors.


顔検出・表情分類・478点Face Landmarkerを読み込みました。
0: anger / 怒り
1: contempt / 軽蔑
2: disgust / 嫌悪
3: fear / 恐れ
4: happiness / 喜び
5: neutral / 無表情
6: sadness / 悲しみ
7: surprise / 驚き


## 3. カメラによるリアルタイム認識

- 赤い `カメラを停止` ボタン：終了
- `画像を保存` ボタン：結果画像を保存
- Notebookの中断ボタンでも終了可能

カメラには1920×1080（1080p）を要求し、そのフレームで顔検出と表情認識を行う。VS Code上のプレビューも1920×1080のフル解像度で送信し、プレビュー更新は最大12 FPSとする。実際にカメラが受理した解像度はボタン横に表示される。

VS CodeではOpenCV/Cocoaの別ウィンドウを使用せず、認識画面をこのセルの出力欄に表示する。カメラ処理はバックグラウンドで動き、セルが完了状態に戻るためボタン操作を受信できる。

`CAMERA_INDEX = None` にすると、AVFoundationのデバイス名から `FaceTime HDカメラ` または `MacBook`カメラを自動選択する。indexが変動してもiPhoneへの自動回避は行わない。

In [6]:
CAMERA_INDEX = 1  # 現在のMacBook内蔵カメラ。名前で自動選択する場合はNone

# VS Codeのセル内に表示する。赤い「カメラを停止」ボタンで終了する。
camera_session = run_camera_in_notebook(camera_index=CAMERA_INDEX)

W0000 00:00:1784278531.411381 4259815 face_landmarker_graph.cc:180] Sets FaceBlendshapesGraph acceleration to xnnpack by default.
I0000 00:00:1784278531.414366 4259815 gl_context.cc:407] GL version: 2.1 (2.1 Metal - 90.5), renderer: Apple M1 Pro
W0000 00:00:1784278531.415627 4259818 inference_feedback_manager.cc:121] Feedback manager requires a model with a single signature inference. Disabling support for feedback tensors.
W0000 00:00:1784278531.423787 4259818 inference_feedback_manager.cc:121] Feedback manager requires a model with a single signature inference. Disabling support for feedback tensors.


## 4. 保存した結果の確認

カメラ画面の `画像を保存` ボタンを押した後に実行する。

In [ ]:
import matplotlib.pyplot as plt

OUTPUT_PATH = PROJECT_DIR / 'MP-0_M25W0243_emotion_result.jpg'
if OUTPUT_PATH.exists():
    saved_image = cv2.imread(str(OUTPUT_PATH))
    plt.figure(figsize=(12, 7))
    plt.imshow(cv2.cvtColor(saved_image, cv2.COLOR_BGR2RGB))
    plt.title(OUTPUT_PATH.name)
    plt.axis('off')
    plt.show()
else:
    print(f'まだ保存画像がありません: {OUTPUT_PATH}')

## 考察

- YuNetは複数の顔と各顔の両目・鼻・口両端の5点を同時に検出できる。
- 5点ランドマークで顔の回転と位置を揃えることで、顔の傾きによる入力のずれを減らす。
- EmotiEffLib公式実装に合わせ、224×224 RGBに変換し、ImageNetの平均・標準偏差で正規化して入力する。
- MediaPipe Face Landmarkerの478点と52 blendshapesから、mouth_open_ratio、jaw_open、brow_raise、brow_furrow、smile、eye_wideを抽出する。
- 顔動作特徴にもEMAを適用し、ゲーム操作の誤触発を減らす。
- 指数移動平均（EMA）と連続確認で、1回だけの誤分類による表示の切り替わりを抑える。
- 最大確率が45%未満、または上位2クラスの差が10ポイント未満の場合は「判定不能」と表示する。
- 顔が小さい、ぼやけている、または極端に暗い・明るい場合も強制分類しない。
- 顔の角度、照明、マスク、顔の大きさによって認識結果が変化する可能性がある。
- 表情分類は外見上の表情を推定するものであり、人の本当の感情を判断するものではない。